# RAG Inference & Evaluasi Notebook

Notebook komprehensif untuk:
1. **Single Query Inference** — tanya bebas, bandingkan 3 metode chunking
2. **Mini Batch Eval** — 5 quick QA × 3 method, hitung BLEU/ROUGE-L + P@k/R@k/MRR
3. **Visualisasi** — bar chart, line chart, similarity heatmap
4. **Chunk Debug** — analisis kenapa chunk A terambil di method X tapi tidak di Y

> Model pipeline di-load sekali per kernel session. Restart kernel jika ingin ganti model.

## 1. Setup & Imports

In [ ]:
import os
import sys
import json
import time
import logging
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

# Tambahkan root project ke path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.rag.pipeline import RAGPipeline, build_pipeline, COLLECTION_NAMES
from src.evaluation.metrics import (
    compute_bleu,
    compute_rouge,
    compute_precision_at_k,
    compute_recall_at_k,
    compute_mrr,
)

# Matplotlib style
sns.set_theme(style='whitegrid', context='notebook', palette='deep')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

print('Setup selesai. ROOT =', ROOT)

## 2. Konfigurasi

In [ ]:
# ── Paths ──
EMBEDDER_PATH   = str(ROOT / "models" / "Qwen3-Embedding-4B")
CHROMA_PATH     = str(ROOT / "data" / "chroma")
QA_GOLD_PATH    = ROOT / "data" / "ground_truth" / "qa_gold_standard_rag_bps_30qa_question_newest.xlsx"
GT_STRICT_PATH  = ROOT / "data" / "ground_truth" / "qa_pairs_strict.json"
GT_LENIENT_PATH = ROOT / "data" / "ground_truth" / "qa_pairs_lenient.json"

# ── Generator config (LOCKED sesuai dokumentasi Qwen3-4B-Instruct-2507) ──
GENERATOR_TYPE  = "hf"
GENERATOR_PATH  = "Qwen/Qwen3-4B-Instruct-2507"  # atau path lokal: ROOT / "models" / "Qwen3-4B-Instruct-2507"
DEFAULT_TOP_K   = 8
MAX_TOKENS      = 1024
TEMPERATURE     = 0.7
TOP_P           = 0.8
TOP_K_GEN       = 20

# ── Method labels ──
METHODS = list(COLLECTION_NAMES.keys())
METHOD_LABELS = {
    "element_based":   "Element-Based",
    "maxmin_semantic": "MaxMin Semantic",
    "recursive":       "Recursive",
}

# ── Quick eval subset (5 QA stabil untuk demo cepat) ──
QUICK_EVAL_IDS = ["Q005", "Q010", "Q011", "Q013", "Q020"]

print('Konfigurasi siap.')
print('Methods:', METHODS)
print('Quick eval IDs:', QUICK_EVAL_IDS)

## 3. Load Pipeline

Memuat embedder + generator + ChromaDB sekali. Ini memakan waktu beberapa menit pertama kali.

In [ ]:
%%time
base_pipeline = build_pipeline(
    chunking_method="element_based",
    embedder_path=EMBEDDER_PATH,
    generator_path=str(GENERATOR_PATH) if isinstance(GENERATOR_PATH, Path) else GENERATOR_PATH,
    generator_type=GENERATOR_TYPE,
    embedder_mode="huggingface",
    chroma_path=CHROMA_PATH,
    top_k=DEFAULT_TOP_K,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k_gen=TOP_K_GEN,
    return_thinking=False,
)
print('\nPipeline berhasil di-load!')
print(f"  Embedder: {type(base_pipeline.embedder).__name__}")
print(f"  Generator: {type(base_pipeline.generator).__name__}")
print(f"  ChromaDB: {CHROMA_PATH}")

## 4. Single Query Inference

Tanya satu pertanyaan → retrieve top-k dari 3 method → generate jawaban → tampilkan side-by-side.

In [ ]:
def run_single_query(query: str, top_k: int = DEFAULT_TOP_K):
    """
    Jalankan inference untuk satu query pada ketiga method chunking.
    Pre-compute embedding sekali, reuse untuk semua method.
    """
    query_vec = base_pipeline.embedder.embed(query)[0]
    results = {}

    for method in METHODS:
        p = RAGPipeline(
            embedder=base_pipeline.embedder,
            generator=base_pipeline.generator,
            chroma_client=base_pipeline.chroma_client,
            chunking_method=method,
            top_k=top_k,
        )
        retrieved = p.retrieve_by_vector(query_vec, k=top_k)
        contexts = [p._format_context(doc) for doc in retrieved]
        raw = p.generator.generate(query, contexts)
        answer = raw[0] if isinstance(raw, tuple) else raw
        results[method] = {
            "answer": answer,
            "chunks": retrieved,
            "n_chunks": len(retrieved),
        }
    return results, query_vec


def render_comparison(query: str, results: dict):
    """Render perbandingan 3 method dalam tabel HTML."""
    rows = []
    for method in METHODS:
        r = results[method]
        ans = r['answer'].replace('\n', '<br>')
        rows.append(f"""
            <tr>
                <td style='font-weight:bold; background:#f0f4f8;'>{METHOD_LABELS[method]}</td>
                <td style='max-width:600px;'>{ans}</td>
                <td style='text-align:center;'>{r['n_chunks']}</td>
            </tr>
        """)

    html = f"""
    <h4>Query: {query}</h4>
    <table style='border-collapse:collapse; width:100%; font-family:sans-serif;'>
        <thead>
            <tr style='background:#1e40af; color:white;'>
                <th style='padding:10px; text-align:left; width:140px;'>Method</th>
                <th style='padding:10px; text-align:left;'>Generated Answer</th>
                <th style='padding:10px; text-align:center; width:80px;'>Chunks</th>
            </tr>
        </thead>
        <tbody>
            {''.join(rows)}
        </tbody>
    </table>
    """
    display(HTML(html))

In [ ]:
# ── Ganti pertanyaan di sini ──
QUERY = "Berapa nilai impor Indonesia pada Agustus 2025?"

results_single, q_vec = run_single_query(QUERY, top_k=DEFAULT_TOP_K)
render_comparison(QUERY, results_single)

### Lihat Retrieved Chunks per Method

In [ ]:
def show_chunks(results: dict, method: str, preview_len: int = 400):
    """Tampilkan chunk yang di-retrieve untuk satu method."""
    chunks = results[method]['chunks']
    print(f"\n{'='*60}")
    print(f"Method: {METHOD_LABELS[method]} ({len(chunks)} chunks)")
    print(f"{'='*60}")
    for i, doc in enumerate(chunks, 1):
        meta = doc.get('metadata', {})
        src = Path(meta.get('source_file', '?')).name
        pages = meta.get('page_numbers', '-')
        dist = doc.get('distance')
        dist_str = f'{dist:.4f}' if dist is not None else '-'
        print(f"\n--- Chunk [{i}] | {src} · hal {pages} · dist {dist_str} ---")
        print(doc['document'][:preview_len].replace('\n', ' ') + '...')

# Tampilkan untuk semua method
for m in METHODS:
    show_chunks(results_single, m)

## 5. Load QA Gold & Ground Truth

In [ ]:
def load_qa_gold(path: Path):
    """Load QA gold dari xlsx."""
    df = pd.read_excel(str(path), sheet_name='qa_gold', dtype=str).fillna('')
    rows = []
    for _, r in df.iterrows():
        qid = str(r.get('query_id', '')).strip()
        if qid:
            rows.append({
                'id': qid,
                'question': str(r['question']).strip(),
                'reference_answer': str(r['gold_answer']).strip(),
            })
    return pd.DataFrame(rows)


def load_ground_truth(path: Path):
    """Load ground truth JSON."""
    with open(path, encoding='utf-8') as f:
        return json.load(f)


qa_gold = load_qa_gold(QA_GOLD_PATH)
gt_strict = load_ground_truth(GT_STRICT_PATH)
gt_lenient = load_ground_truth(GT_LENIENT_PATH)

# Lookup dicts
gt_strict_lookup = {item['id']: item for item in gt_strict}
gt_lenient_lookup = {item['id']: item for item in gt_lenient}

print(f'QA Gold loaded: {len(qa_gold)} queries')
print(f'Strict GT: {len(gt_strict)} queries, {sum(len(item.get("relevant_chunk_ids",{}).get(m,[])) for item in gt_strict for m in METHODS)} chunks')
print(f'Lenient GT: {len(gt_lenient)} queries')

## 6. Mini Batch Eval

Evaluasi 5 quick QA × 3 method. Hitung BLEU, ROUGE-L, P@k, R@k, MRR.

In [ ]:
def run_mini_eval(qa_df: pd.DataFrame, gt_lookup: dict, top_k: int = DEFAULT_TOP_K, relevance_mode='strict'):
    """
    Mini batch eval untuk subset QA.
    Returns DataFrame dengan kolom: query_id, method, question, bleu, rouge_l, precision, recall, mrr
    """
    rows = []
    total = len(qa_df) * len(METHODS)
    step = 0

    # Pre-compute embeddings
    q_embeddings = {}
    for _, row in qa_df.iterrows():
        qid = row['id']
        q_embeddings[qid] = base_pipeline.embedder.embed(row['question'])[0]

    for _, row in qa_df.iterrows():
        qid = row['id']
        question = row['question']
        reference = row['reference_answer']
        gt_item = gt_lookup.get(qid)
        q_vec = q_embeddings[qid]

        for method in METHODS:
            step += 1
            print(f"\r[{step}/{total}] {qid} {method:<20}", end='', flush=True)

            p = RAGPipeline(
                embedder=base_pipeline.embedder,
                generator=base_pipeline.generator,
                chroma_client=base_pipeline.chroma_client,
                chunking_method=method,
                top_k=top_k,
            )

            # Retrieve
            retrieved = p.retrieve_by_vector(q_vec, k=top_k)
            retrieved_ids = [doc.get('id', '') for doc in retrieved]

            # Retrieval metrics
            if gt_item:
                rel_all = gt_item.get('relevant_chunk_ids', {})
                rel_ids = rel_all.get(method, []) if isinstance(rel_all, dict) else rel_all
                precision = compute_precision_at_k(retrieved_ids, rel_ids, top_k) if rel_ids else None
                recall = compute_recall_at_k(retrieved_ids, rel_ids, top_k) if rel_ids else None
                mrr = compute_mrr(retrieved_ids, rel_ids) if rel_ids else None
            else:
                precision = recall = mrr = None

            # Generate
            contexts = [p._format_context(doc) for doc in retrieved]
            raw = p.generator.generate(question, contexts)
            answer = raw[0] if isinstance(raw, tuple) else raw

            # Generation metrics
            bleu = compute_bleu(answer, reference) if answer else 0.0
            rouge = compute_rouge(answer, reference, 'rougeL', 'recall') if answer else 0.0

            rows.append({
                'query_id': qid,
                'method': METHOD_LABELS[method],
                'question': question,
                'answer': answer,
                'reference': reference,
                'bleu': round(bleu, 4),
                'rouge_l': round(rouge, 4),
                'precision_at_k': round(precision, 4) if precision is not None else None,
                'recall_at_k': round(recall, 4) if recall is not None else None,
                'mrr': round(mrr, 4) if mrr is not None else None,
            })

    print()  # newline
    return pd.DataFrame(rows)


# Jalankan evaluasi untuk quick subset
qa_subset = qa_gold[qa_gold['id'].isin(QUICK_EVAL_IDS)]
print(f'Mini eval: {len(qa_subset)} queries x {len(METHODS)} methods = {len(qa_subset)*len(METHODS)} runs')

eval_df = run_mini_eval(qa_subset, gt_strict_lookup, top_k=DEFAULT_TOP_K, relevance_mode='strict')
display(eval_df.head(10))

In [ ]:
# Ringkasan agregat per method
summary = eval_df.groupby('method').agg(
    n_queries=('bleu', 'count'),
    mean_bleu=('bleu', 'mean'),
    mean_rouge_l=('rouge_l', 'mean'),
    mean_precision=('precision_at_k', 'mean'),
    mean_recall=('recall_at_k', 'mean'),
    mean_mrr=('mrr', 'mean'),
).round(4)
display(summary)

## 7. Visualisasi: Bar Chart Generation Metrics (BLEU & ROUGE-L)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

methods_ordered = [METHOD_LABELS[m] for m in METHODS]
bleu_vals = [summary.loc[m, 'mean_bleu'] for m in methods_ordered]
rouge_vals = [summary.loc[m, 'mean_rouge_l'] for m in methods_ordered]

x = np.arange(len(methods_ordered))
width = 0.35

bars1 = ax.bar(x - width/2, bleu_vals, width, label='BLEU', color='#2563eb')
bars2 = ax.bar(x + width/2, rouge_vals, width, label='ROUGE-L Recall', color='#16a34a')

ax.set_ylabel('Score')
ax.set_title(f'Mean BLEU & ROUGE-L per Chunking Method (top-k={DEFAULT_TOP_K})')
ax.set_xticks(x)
ax.set_xticklabels(methods_ordered)
ax.legend()
ax.set_ylim(0, 1.0)

# Annotate bars
for bar in bars1:
    h = bar.get_height()
    ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    h = bar.get_height()
    ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 8. Visualisasi: Bar Chart Retrieval Metrics (P@k, R@k, MRR)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

p_vals = [summary.loc[m, 'mean_precision'] for m in methods_ordered]
r_vals = [summary.loc[m, 'mean_recall'] for m in methods_ordered]
mrr_vals = [summary.loc[m, 'mean_mrr'] for m in methods_ordered]

x = np.arange(len(methods_ordered))
width = 0.25

bars1 = ax.bar(x - width, p_vals, width, label=f'P@{DEFAULT_TOP_K}', color='#2563eb')
bars2 = ax.bar(x, r_vals, width, label=f'R@{DEFAULT_TOP_K}', color='#16a34a')
bars3 = ax.bar(x + width, mrr_vals, width, label='MRR', color='#9333ea')

ax.set_ylabel('Score')
ax.set_title(f'Retrieval Metrics per Chunking Method (Strict GT, top-k={DEFAULT_TOP_K})')
ax.set_xticks(x)
ax.set_xticklabels(methods_ordered)
ax.legend()
ax.set_ylim(0, 1.0)

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                        xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 9. Visualisasi: Line Chart — Retrieval Metrics Across Top-K

Baca data dari folder `results/` jika tersedia, atau jalankan mini eval untuk top-k 1–5.

In [ ]:
# Coba baca dari hasil Streamlit batch eval yang sudah ada
results_dir = ROOT / "results" / "RTX 5060 Ti 16GB" / "generation_eval_streamlit"
csv_files = sorted(results_dir.glob("eval_strict_*.csv"), reverse=True) if results_dir.exists() else []

if csv_files:
    # Baca semua file dan agregat per top-k
    top_k_data = defaultdict(lambda: defaultdict(list))
    for fpath in csv_files:
        df = pd.read_csv(fpath)
        # Extract top-k dari nama file (e.g., eval_strict_..._top5.csv)
        fname = fpath.name
        if 'top' in fname:
            k_str = fname.split('top')[-1].replace('.csv', '')
            try:
                k = int(k_str)
                for _, row in df.iterrows():
                    method = row.get('method', '')
                    if method and pd.notna(row.get('precision_at_k')) and str(row['precision_at_k']) not in ('OOM', 'N/A', ''):
                        try:
                            top_k_data[method][k].append({
                                'precision': float(row['precision_at_k']),
                                'recall': float(row['recall_at_k']),
                                'mrr': float(row['mrr']),
                            })
                        except (ValueError, TypeError):
                            pass
            except ValueError:
                pass

    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    metrics = ['precision', 'recall', 'mrr']
    titles = [f'P@k', f'R@k', 'MRR']
    colors = ['#2563eb', '#16a34a', '#9333ea']

    for ax, metric, title in zip(axes, metrics, titles):
        for method_label in methods_ordered:
            if method_label in top_k_data:
                ks = sorted(top_k_data[method_label].keys())
                means = [np.mean([r[metric] for r in top_k_data[method_label][k]]) for k in ks]
                ax.plot(ks, means, marker='o', label=method_label, linewidth=2)
        ax.set_xlabel('Top-K')
        ax.set_ylabel('Score')
        ax.set_title(title)
        ax.legend(fontsize=8)
        ax.set_ylim(0, 1.0)
        ax.grid(True, alpha=0.3)

    plt.suptitle('Retrieval Metrics vs Top-K (Strict GT)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  Tidak ada data batch eval di results/. Jalankan batch eval di Streamlit terlebih dahulu, atau modifikasi cell ini untuk compute inline.")

## 10. Visualisasi: Similarity Heatmap — Query vs Retrieved Chunks

Tunjukkan cosine similarity antara query embedding dan chunk embeddings yang di-retrieve, per method.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

QUERY_HEATMAP = "Berapa nilai impor Indonesia pada Agustus 2025?"  # ganti query di sini
TOP_K_HEATMAP = 5

q_vec_hm = base_pipeline.embedder.embed(QUERY_HEATMAP)[0].reshape(1, -1)

heatmap_data = []
yticklabels = []

for method in METHODS:
    p = RAGPipeline(
        embedder=base_pipeline.embedder,
        generator=base_pipeline.generator,
        chroma_client=base_pipeline.chroma_client,
        chunking_method=method,
        top_k=TOP_K_HEATMAP,
    )
    retrieved = p.retrieve_by_vector(q_vec_hm.reshape(-1), k=TOP_K_HEATMAP)

    for i, doc in enumerate(retrieved):
        # Compute cosine similarity
        # Note: ChromaDB distance might not be raw cosine, so we re-compute
        # We don't have pre-computed chunk embeddings, but we can use the distance field
        dist = doc.get('distance', 0)
        sim = 1 - dist  # ChromaDB default uses cosine distance = 1 - cosine_similarity
        heatmap_data.append(sim)
        preview = doc['document'][:60].replace('\n', ' ')
        yticklabels.append(f"[{METHOD_LABELS[method][:3]}] {preview}...")

# Reshape untuk heatmap: 3 methods x top_k chunks
data_matrix = np.array(heatmap_data).reshape(len(METHODS), TOP_K_HEATMAP)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    data_matrix,
    annot=True,
    fmt='.3f',
    cmap='YlOrRd',
    xticklabels=[f'Chunk {i+1}' for i in range(TOP_K_HEATMAP)],
    yticklabels=[METHOD_LABELS[m] for m in METHODS],
    ax=ax,
    vmin=0, vmax=1,
    cbar_kws={'label': 'Cosine Similarity'}
)
ax.set_title(f'Query: "{QUERY_HEATMAP}" — Similarity to Retrieved Chunks')
plt.tight_layout()
plt.show()

## 11. Chunk Debug View

Untuk satu query, tampilkan chunk ID, similarity score, dan preview teks dari ketiga method sekaligus — membantu analisis kenapa method A retrieve chunk X tapi method B tidak.

In [ ]:
QUERY_DEBUG = "Berapa nilai impor Indonesia pada Agustus 2025?"
TOP_K_DEBUG = 5

q_vec_dbg = base_pipeline.embedder.embed(QUERY_DEBUG)[0]

debug_rows = []
for method in METHODS:
    p = RAGPipeline(
        embedder=base_pipeline.embedder,
        generator=base_pipeline.generator,
        chroma_client=base_pipeline.chroma_client,
        chunking_method=method,
        top_k=TOP_K_DEBUG,
    )
    retrieved = p.retrieve_by_vector(q_vec_dbg, k=TOP_K_DEBUG)
    for rank, doc in enumerate(retrieved, 1):
        dist = doc.get('distance')
        sim = 1 - dist if dist is not None else None
        meta = doc.get('metadata', {})
        debug_rows.append({
            'method': METHOD_LABELS[method],
            'rank': rank,
            'chunk_id': doc.get('id', '?'),
            'similarity': round(sim, 4) if sim is not None else None,
            'source': Path(meta.get('source_file', '?')).name,
            'pages': meta.get('page_numbers', '-'),
            'preview': doc['document'][:120].replace('\n', ' ') + '...',
        })

debug_df = pd.DataFrame(debug_rows)
display(debug_df.style.set_properties(**{'font-size': '11px'}))

# Highlight: tampilkan chunk ID unik per method
print('\n--- Chunk ID unik per method ---')
for method in METHODS:
    mids = debug_df[debug_df['method'] == METHOD_LABELS[method]]['chunk_id'].tolist()
    print(f"{METHOD_LABELS[method]}: {mids}")

---
## Tips Penggunaan

- **Ganti query**: Edit variabel `QUERY` di Cell 4 atau `QUERY_HEATMAP` / `QUERY_DEBUG` di cell visualisasi.
- **Ganti top-k**: Ubah `DEFAULT_TOP_K` di Cell 2 atau parameter `top_k=` di fungsi spesifik.
- **Full eval 30 QA**: Ganti `QUICK_EVAL_IDS` dengan `qa_gold['id'].tolist()` di Cell 6.
- **Lenient GT**: Ganti `gt_strict_lookup` ke `gt_lenient_lookup` di Cell 6.
- **Save hasil**: Gunakan `eval_df.to_csv('hasil_eval.csv', index=False)` setelah Cell 6.